# Дообучение нейронки EyePhoto

Перед запуском справа в панели **Settings**: *Accelerator* — **GPU T4 x2** или **GPU P100**, *Internet* — **On**. Глаза: либо слева **+ Add Input** → ваш датасет с зипками, либо ссылки на Google Диск в ячейке «где лежат глаза».

Запускать через **Save Version → Save & Run All (Commit)**: тогда обучение идёт на серверах Kaggle, даже если закрыть браузер. Результат появится во вкладке **Output** этой версии.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# скрипты обучения из репозитория программы
import os, urllib.request
BASE = 'https://raw.githubusercontent.com/chsts0/eyephoto/main/training/'
os.makedirs('/kaggle/working/training', exist_ok=True)
for f in ['train.py', 'lama_model.py', 'safe_ckpt.py']:
    urllib.request.urlretrieve(BASE + f, '/kaggle/working/training/' + f)
print(os.listdir('/kaggle/working/training'))

In [ ]:
# где лежат глаза.
# Вариант 1: зипки загружены в датасет Kaggle и подключены через + Add Input — ничего не трогать.
# Вариант 2: зипки лежат на Google Диске. Вставьте ссылки ниже (доступ «Все, у кого есть ссылка»),
# можно ссылку на целую папку. Нужен Internet: On.
DRIVE = [
    # 'https://drive.google.com/file/d/.../view?usp=sharing',
    # 'https://drive.google.com/drive/folders/...',
]
import glob, os, subprocess, zipfile
if DRIVE:
    subprocess.run(['pip', 'install', '-q', 'gdown'], check=True)
    import gdown
    DATA = '/kaggle/working/data'
    os.makedirs(DATA, exist_ok=True)
    for url in DRIVE:
        if '/folders/' in url:
            gdown.download_folder(url, output=DATA, quiet=False)
        else:
            gdown.download(url, output=DATA + '/', fuzzy=True, quiet=False)
    for z in glob.glob(DATA + '/**/*.zip', recursive=True):
        zipfile.ZipFile(z).extractall(z[:-4])
        os.remove(z)
else:
    # из подключённых входов берём тот, где лежат глаза
    DATA = next((d for d in sorted(glob.glob('/kaggle/input/*')) if glob.glob(d + '/**/eye_*_mask.png', recursive=True)), '/kaggle/input')
n = len(glob.glob(DATA + '/**/eye_*_mask.png', recursive=True))
print(DATA, '— глаз:', n)
assert n > 0, 'Не нашёл глаз: подключите датасет через + Add Input или впишите ссылки в DRIVE'


In [ ]:
# с чего начинать: если подключён результат прошлого обучения (+ Add Input → Your Work →
# этот блокнот), продолжаем с его лучшей модели. Иначе — с исходной big-lama.
import glob
INIT = next(iter(sorted(glob.glob('/kaggle/input/**/best_gen.pt', recursive=True))), '')
print('Старт с:', INIT or 'исходной модели')
# обучение: ~9 часов (Kaggle даёт до 12 за сессию), полная точность, батч 2
!python /kaggle/working/training/train.py --data "$DATA" --out /kaggle/working/out --hours 9 --batch 2 --cache /kaggle/working/cache --init "$INIT"


In [ ]:
# итог: график ошибки и последние сравнения (слева направо: вход, старая модель, новая, как было)
import pandas as pd, glob
from IPython.display import Image, display
print(pd.read_csv('/kaggle/working/out/log.csv'))
for p in sorted(glob.glob('/kaggle/working/out/samples/*.jpg'))[-2:]:
    print(p); display(Image(p, width=1000))

In [ ]:
# чтобы во вкладке Output не лежали лишние 1,5 ГБ: оставляем модель, примеры и лог
import shutil, os
shutil.rmtree('/kaggle/working/cache', ignore_errors=True)
for f in ['last.pt']:
    p = '/kaggle/working/out/' + f
    if os.path.exists(p): os.remove(p)
print(os.listdir('/kaggle/working/out'))